### Human Airway Smooth Muscle Cell Responses to Asthma Medications
This assignment is using the GEO GSE52778 (airway RNA-seq) dataset. This dataset contains RNA sequencing data from human airway smooth muscle cells under four different asthma treatment conditions: treated with  Dexamethasone, Albuterol, left untreated, or treated with both Dexamethasone + Albuterol.

### Research Question: 
How do different asthma treatments affect gene expression in human airway smooth muscle cells?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [ ]:
url = ("https://www.ncbi.nlm.nih.gov/geo/download/"
       "?acc=GSE52778&format=file&file=GSE52778_All_Sample_FPKM_Matrix.txt.gz")
fpkm = pd.read_csv(url, sep=r"\s+", compression="gzip")

fpkm.head()
fpkm.shape
fpkm.columns
fpkm.info()

for column in fpkm.columns:
    print(column)

In [ ]:
sample_columns = [
    "Dex_LL14",
    "Dex_LL06",
    "Dex_LL02",
    "Dex_LL10",
    "Alb_LL07",
    "Alb_LL03",
    "Alb_LL11",
    "Alb_LL15",
    "Untreated_LL09",
    "Untreated_LL05",
    "Untreated_LL13",
    "Untreated_LL01",
    "Alb_Dex_LL08",
    "Alb_Dex_LL04",
    "Alb_Dex_LL12",
    "Alb_Dex_LL16"
]

missing_columns = [
    column for column in sample_columns
    if column not in fpkm.columns
]

print(missing_columns)

In [ ]:
expression = fpkm[sample_columns]
expression.head()
expression.shape

In [ ]:
log_expression = np.log2(expression + 1)
log_expression.head()
log_expression.describe()

expression_for_pca = log_expression.T
expression_for_pca.shape

In [ ]:
pca = PCA(n_components=2)
pca_results = pca.fit_transform(expression_for_pca)

pca_df = pd.DataFrame(
    pca_results,
    columns=["PC1", "PC2"],
    index=sample_columns
)

pca_df

In [ ]:
print(pca.explained_variance_ratio_)

explained = pca.explained_variance_ratio_ * 100

print(f"PC1 explains {explained[0]:.2f}% of the variance")
print(f"PC2 explains {explained[1]:.2f}% of the variance")
print(f"Together they explain {explained.sum():.2f}% of the variance")

In [ ]:
metadata = pd.DataFrame({
    "Sample": sample_columns,
    "Treatment": [
        "Dexamethasone",
        "Dexamethasone",
        "Dexamethasone",
        "Dexamethasone",
        "Albuterol",
        "Albuterol",
        "Albuterol",
        "Albuterol",
        "No treatment",
        "No treatment",
        "No treatment",
        "No treatment",
        "Dexamethasone + Albuterol",
        "Dexamethasone + Albuterol",
        "Dexamethasone + Albuterol",
        "Dexamethasone + Albuterol"
    ]
})

metadata

In [ ]:
pca_df = pca_df.reset_index().rename(columns={"index": "Sample"})
pca_df = pca_df.merge(metadata, on="Sample")
pca_df

In [ ]:
for treatment in pca_df["Treatment"].unique():
    subset = pca_df[pca_df["Treatment"] == treatment]
    plt.scatter(
        subset["PC1"],
        subset["PC2"],
        label=treatment
    )

plt.xlabel(f"PC1 ({explained[0]:.2f}% variance)")
plt.ylabel(f"PC2 ({explained[1]:.2f}% variance)")
plt.title("PCA of Human Airway Smooth Muscle Gene Expression")
plt.legend()
plt.show()

I used PCA to explore whether the samples receiving different asthma treatments showed differences in their overall gene expression. A log transformation was done (took the log base 2) of each gene expression value to compress the scale of the data and make them more balanced, so extremely large values didn't have a large influence on the final results.

### Interpretation of the Result
According to the PCA, both the gene expression of Dexamethasone treated samples and untreated samples showed little variation in gene expression, and clearly differed from each other along PC1. The Albuterol and Dexamethasone + Albuterol treated samples, however, showed greater variation in gene expression and less distinct clustering. Overall, these results suggest that different asthma treatments can be associated with differences in gene expression patterns in human airway smooth muscle cells.